# Jyske Bank statement – explore include / exclude

This notebook is for deciding **what should enter the dashboard calculations** and what is just money moving between your own accounts.

Jyske is not Revolut: most card spend already lives on Revolut. This checking account is mostly:

- salary and benefits in
- top-ups out to Revolut (already counted as Revolut spend)
- transfers to/from **Opsparingskonto** (savings, not spend)
- a smaller set of bills paid directly from Jyske (mortgage, Tryg, parking, gym, union, car tax)

Tweak the `bucket()` rules in section 6, then re-run the summary cells. We will wire the parser after this looks right.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

CSV_PATH = Path(
    "/Users/mehdiordikhani/Library/Mobile Documents/com~apple~Numbers/Documents/Mehdi Ordikhani_2026-01-01-2026-08-24.csv"
)
print("csv_path:", CSV_PATH)
print("exists:", CSV_PATH.exists())

csv_path: /Users/mehdiordikhani/Library/Mobile Documents/com~apple~Numbers/Documents/Mehdi Ordikhani_2026-01-01-2026-08-24.csv
exists: True


## 1. Load

Export layout (different from Revolut):

- separator `;`
- date `DD.MM.YYYY`
- amount like `-4,021.00` (comma thousands, period decimals)
- Jyske already ships `MainCategory` / `Category`

In [2]:
raw = pd.read_csv(CSV_PATH, sep=";")
print("rows:", len(raw), "| cols:", list(raw.columns))
raw.head(10)

rows: 204 | cols: ['Date', 'Text', 'Amount', 'Balance', 'Reconciled', 'AccountNumber', 'AccountName', 'MainCategory', 'Category', 'Comment']


,Date,Text,Amount,Balance,Reconciled,AccountNumber,AccountName,MainCategory,Category,Comment
0,24.08.2026,VD REVOLUT**8260*,"-4,021.00",132.30,NaN,5030 1360462,Mehdi Ordikhani,Loan and debt,Interest and fees,NaN
1,17.08.2026,VD REVOLUT**8260*,"-4,392.00","4,153.30",NaN,5030 1360462,Mehdi Ordikhani,Loan and debt,Interest and fees,NaN
2,12.08.2026,MobilePay Boozt.com,"1,553.00","8,545.30",NaN,5030 1360462,Mehdi Ordikhani,Income,Other income,NaN
3,12.08.2026,MobilePay Boozt.com,"1,731.00","6,992.30",NaN,5030 1360462,Mehdi Ordikhani,Income,Other income,NaN
4,11.08.2026,VD REVOLUT**8260*,"-3,869.00","5,261.30",NaN,5030 1360462,Mehdi Ordikhani,Loan and debt,Interest and fees,NaN
5,11.08.2026,Opsparingskonto,"4,923.00","9,130.30",NaN,5030 1360462,Mehdi Ordikhani,Income,Other income,NaN
6,11.08.2026,3084302067637721,145.41,"4,207.30",NaN,5030 1360462,Mehdi Ordikhani,Income,Other income,NaN
7,10.08.2026,VD REVOLUT**8260*,"-5,000.00","4,061.89",NaN,5030 1360462,Mehdi Ordikhani,Loan and debt,Interest and fees,NaN
8,10.08.2026,Opsparingskonto,"7,000.00","9,061.89",NaN,5030 1360462,Mehdi Ordikhani,Income,Other income,NaN
9,10.08.2026,Opsparingskonto,"1,574.00","2,061.89",NaN,5030 1360462,Mehdi Ordikhani,Income,Other income,NaN


## 2. Parse dates and amounts

In [3]:
def parse_jyske_amount(value) -> float:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return np.nan
    s = str(value).strip().replace("\xa0", "").replace(" ", "").replace(",", "")
    if not s:
        return np.nan
    return float(s)


df = raw.copy()
df.columns = [c.strip() for c in df.columns]
df["date"] = pd.to_datetime(df["Date"], format="%d.%m.%Y", errors="coerce")
df["amount"] = df["Amount"].map(parse_jyske_amount)
df["balance"] = df["Balance"].map(parse_jyske_amount)
df["text"] = df["Text"].astype(str).str.strip()
df["main_category"] = df["MainCategory"].astype(str).str.strip()
df["category"] = df["Category"].astype(str).str.strip()
df["month"] = df["date"].dt.to_period("M").astype(str)

print("date range:", df["date"].min().date(), "→", df["date"].max().date())
print("parsed NaT dates:", int(df["date"].isna().sum()), "| NaN amounts:", int(df["amount"].isna().sum()))
print("net of all rows (should be near 0 aside from start/end balance):", f"{df['amount'].sum():,.2f}")
df[["date", "text", "amount", "balance", "main_category", "category"]].head()

date range: 2026-01-02 → 2026-08-24
parsed NaT dates: 0 | NaN amounts: 0
net of all rows (should be near 0 aside from start/end balance): -2,043.75


,date,text,amount,balance,main_category,category
0,2026-08-24,VD REVOLUT**8260*,"-4,021.00",132.30,Loan and debt,Interest and fees
1,2026-08-17,VD REVOLUT**8260*,"-4,392.00","4,153.30",Loan and debt,Interest and fees
2,2026-08-12,MobilePay Boozt.com,"1,553.00","8,545.30",Income,Other income
3,2026-08-12,MobilePay Boozt.com,"1,731.00","6,992.30",Income,Other income
4,2026-08-11,VD REVOLUT**8260*,"-3,869.00","5,261.30",Loan and debt,Interest and fees


## 3. File shape

One account in this export. `Reconciled` and `Comment` are unused.

In [4]:
print("accounts:", df["AccountNumber"].unique().tolist(), df["AccountName"].unique().tolist())
print("reconciled:", df["Reconciled"].value_counts(dropna=False).to_dict())
print("comment non-null:", int(df["Comment"].notna().sum()))
print("unique texts:", df["text"].nunique())
print("outflows:", int((df["amount"] < 0).sum()), f"{df.loc[df['amount'] < 0, 'amount'].sum():,.2f}")
print("inflows:", int((df["amount"] > 0).sum()), f"{df.loc[df['amount'] > 0, 'amount'].sum():,.2f}")

accounts: ['5030 1360462'] ['Mehdi Ordikhani']
reconciled: {nan: 204}
comment non-null: 0
unique texts: 31
outflows: 127 -770,189.57
inflows: 77 768,145.82


## 4. Jyske's own categories

Useful as a hint, **not** as the final include/exclude. Example: Revolut top-ups are labelled *Loan and debt / Interest and fees*, which is wrong for our dashboard.

In [5]:
(
    df.groupby(["main_category", "category"], dropna=False)["amount"]
    .agg(n="count", sum="sum")
    .sort_values("sum")
)

n         sum
main_category          category                                        
Savings and investment Savings                           14 -392,032.00
Loan and debt          Interest and fees                 75 -231,272.00
Other                  Other expense                     15  -96,916.00
Home                   Mortgage loan                      2  -27,362.41
Insurance              Insurance (Other)                  4  -14,220.94
                       Union and unemployment insurance   3   -3,307.50
Transportation         Parking                            8   -1,796.72
                       Motor vehicle tax                  1   -1,630.00
Other                  Other (Transfer)                   3     -874.00
Leisure                Sport and leisure activities       2     -778.00
Income                 Social security                    4    8,057.00
                       Pay, benefits and pension          7  374,012.80
                       Other income                      66  386,076.02

## 5. Group by text

This is the real filter surface — 30-ish distinct descriptions.

In [6]:
by_text = (
    df.groupby("text", dropna=False)
    .agg(
        n=("amount", "count"),
        sum=("amount", "sum"),
        min_date=("date", "min"),
        max_date=("date", "max"),
        main_category=("main_category", lambda s: ", ".join(sorted(set(s.astype(str))))),
        category=("category", lambda s: ", ".join(sorted(set(s.astype(str))))),
    )
    .sort_values("sum")
)
by_text

,n,sum,min_date,max_date,main_category,category
text,,,,,,
VD REVOLUT**8260*,59,"-204,465.00",2026-02-02,2026-08-24,Loan and debt,Interest and fees
Opsparingskonto,58,"-190,454.00",2026-01-02,2026-08-11,"Income, Savings and investment","Other income, Savings"
Mehdi Ordikhani,2,"-63,000.00",2026-01-05,2026-03-02,Other,Other expense
BS JYSKE REALKREDIT,2,"-27,362.41",2026-03-31,2026-06-30,Home,Mortgage loan
VD Revolut**8260*,10,"-21,590.00",2026-01-02,2026-07-13,Loan and debt,Interest and fees
To Tryg Forsikring,2,"-13,364.86",2026-07-13,2026-07-13,Insurance,Insurance (Other)
Leo - Oskarkonto,4,"-9,719.00",2026-01-12,2026-07-20,Other,Other expense
5030 1360470,1,"-9,470.00",2026-04-13,2026-04-13,Other,Other expense
Lara - Oskarkonto,3,"-7,400.00",2026-01-12,2026-07-20,Other,Other expense


## 6. Proposed buckets — **edit this cell**

Starting proposal:

| Bucket | Meaning | Examples |
|---|---|---|
| `exclude_revolut_topup` | Already spent on Revolut — including it here would double-count | `VD REVOLUT**8260*` |
| `exclude_savings` | Internal savings moves, not spend/income | `Opsparingskonto` |
| `exclude_internal` | Transfers between own accounts / unlabeled inflows | `Overførsel`, `Mehdi Ordikhani`, `From Lunar`, `5030 1360470` |
| `income` | Real incoming money | `Lønoverførsel`, `Børne- og Ungeydelse` |
| `refund` | Money back (not salary) | `MobilePay Boozt.com` |
| `expense` | Bills paid from Jyske that are not on Revolut | Tryg, mortgage, parking, gym, union, car tax |
| `review` | Decide with a closer look | kids' Oskarkonto, personal MobilePay, home tax |

Change patterns below, then re-run from here.

In [7]:
def bucket(text: str) -> str:
    t = str(text).casefold()

    if "revolut" in t:
        return "exclude_revolut_topup"
    if "opsparing" in t:
        return "exclude_savings"
    if t in {"overførsel", "mehdi ordikhani", "from lunar"} or t.replace(" ", "").startswith("5030"):
        return "exclude_internal"
    if t.startswith("lønoverførsel") or "børne- og ungeydelse" in t:
        return "income"
    if "boozt" in t:
        return "refund"

    expense_needles = (
        "tryg",
        "letsikring",
        "jyske realkredit",
        "parkeringslauget",
        "pure gym",
        "skattestyrelsen motor",
        "ida div",
        "akademikernes",
        "omkostninger, netbank",
    )
    if any(n in t for n in expense_needles):
        return "expense"

    return "review"


df["bucket"] = df["text"].map(bucket)
df["bucket"].value_counts()

bucket
exclude_revolut_topup    70
exclude_savings          58
expense                  25
review                   19
exclude_internal         16
income                   11
refund                    5
Name: count, dtype: int64

## 7. Totals under that proposal

In [8]:
summary = (
    df.groupby("bucket")["amount"]
    .agg(n="count", sum="sum")
    .reindex(
        [
            "expense",
            "income",
            "refund",
            "review",
            "exclude_revolut_topup",
            "exclude_savings",
            "exclude_internal",
        ]
    )
)
summary.loc["included_net (exp+inc+ref)"] = [
    int(df["bucket"].isin(["expense", "income", "refund"]).sum()),
    df.loc[df["bucket"].isin(["expense", "income", "refund"]), "amount"].sum(),
]
summary

,n,sum
bucket,,
expense,25.00,"-49,353.57"
income,11.00,"382,069.80"
refund,5.00,"6,680.00"
review,19.00,"-9,615.59"
exclude_revolut_topup,70.00,"-231,014.00"
exclude_savings,58.00,"-190,454.00"
exclude_internal,16.00,"89,643.61"
included_net (exp+inc+ref),41.00,"339,396.23"


In [9]:
included = df[df["bucket"].isin(["expense", "income", "refund"])].copy()
print("Proposed Jyske expense (absolute):", f"{included.loc[included.bucket.eq('expense'), 'amount'].sum():,.2f}")
print("Proposed Jyske income:", f"{included.loc[included.bucket.eq('income'), 'amount'].sum():,.2f}")
print("Proposed Jyske refunds:", f"{included.loc[included.bucket.eq('refund'), 'amount'].sum():,.2f}")

(
    included[included["bucket"].eq("expense")]
    .groupby("text")["amount"]
    .agg(n="count", sum="sum")
    .sort_values("sum")
)

Proposed Jyske expense (absolute): -49,353.57
Proposed Jyske income: 382,069.80
Proposed Jyske refunds: 6,680.00


,n,sum
text,,
BS JYSKE REALKREDIT,2,"-27,362.41"
To Tryg Forsikring,2,"-13,364.86"
BS PARKERINGSLAUGET PARKKANTEN,8,"-1,796.72"
Ida Div.Kontingent,2,"-1,756.50"
BS SKATTESTYRELSEN MOTOR OPKRÆVNING,1,"-1,630.00"
Akademikernes,1,"-1,551.00"
Letsikring af barn ved dø,2,-856.08
BS PURE GYM DENMARK A/S,2,-778.00
"Omkostninger, Netbank og Mobilbank",5,-258.00


## 8. Inspect each bucket

If a row is in the wrong bucket, add/remove a pattern in `bucket()` and re-run from section 6.

In [10]:
cols = ["date", "text", "amount", "main_category", "category", "bucket"]

def show(b: str):
    sub = df.loc[df["bucket"].eq(b), cols].sort_values(["date", "text"], ascending=[False, True])
    print(f"\n=== {b}  n={len(sub)}  sum={sub['amount'].sum():,.2f} ===")
    return sub

show("review")


=== review  n=19  sum=-9,615.59 ===


,date,text,amount,main_category,category,bucket
6,2026-08-11,3084302067637721,145.41,Income,Other income,review
27,2026-07-23,MobilePay Anna Sewell Møller,-32.00,Other,Other (Transfer),review
29,2026-07-23,MobilePay Anna Sewell Møller,-281.00,Other,Other (Transfer),review
31,2026-07-21,MobilePay Anna Sewell Møller,-561.00,Other,Other (Transfer),review
34,2026-07-20,Lara - Oskarkonto,"-1,200.00",Other,Other expense,review
33,2026-07-20,Leo - Oskarkonto,"-1,200.00",Other,Other expense,review
42,2026-07-13,Karoline Bianca Dose,"14,000.00",Income,Other income,review
62,2026-06-26,MobilePay Karoline Bianca,"1,000.00",Income,Other income,review
99,2026-05-06,Iran Consulate,-384.00,Other,Other expense,review
115,2026-04-28,Mads Beich: Lara s event,"-1,202.00",Other,Other expense,review


In [11]:
show("expense")


=== expense  n=25  sum=-49,353.57 ===


,date,text,amount,main_category,category,bucket
11,2026-08-05,Ida Div.Kontingent,-878.25,Insurance,Union and unemployment insurance,expense
16,2026-08-03,BS PARKERINGSLAUGET PARKKANTEN,-224.59,Transportation,Parking,expense
22,2026-07-31,"Omkostninger, Netbank og Mobilbank",-56.00,Loan and debt,Interest and fees,expense
43,2026-07-13,To Tryg Forsikring,"-7,442.83",Insurance,Insurance (Other),expense
44,2026-07-13,To Tryg Forsikring,"-5,922.03",Insurance,Insurance (Other),expense
54,2026-07-01,Akademikernes,"-1,551.00",Insurance,Union and unemployment insurance,expense
53,2026-07-01,BS PARKERINGSLAUGET PARKKANTEN,-224.59,Transportation,Parking,expense
58,2026-06-30,BS JYSKE REALKREDIT,"-13,674.71",Home,Mortgage loan,expense
73,2026-06-01,BS PARKERINGSLAUGET PARKKANTEN,-147.14,Transportation,Parking,expense
80,2026-05-29,"Omkostninger, Netbank og Mobilbank",-2.00,Loan and debt,Interest and fees,expense


In [12]:
show("income")


=== income  n=11  sum=382,069.80 ===


,date,text,amount,main_category,category,bucket
19,2026-07-31,Lønoverførsel,"50,571.00",Income,"Pay, benefits and pension",income
36,2026-07-20,Børne- og Ungeydelse,"2,319.00",Income,Social security,income
57,2026-06-30,Lønoverførsel,"53,075.00",Income,"Pay, benefits and pension",income
79,2026-05-29,Børne- og Ungeydelse,"1,100.00",Income,Social security,income
78,2026-05-29,Lønoverførsel,"58,066.80",Income,"Pay, benefits and pension",income
112,2026-04-30,Lønoverførsel,"53,075.00",Income,"Pay, benefits and pension",income
127,2026-04-20,Børne- og Ungeydelse,"2,319.00",Income,Social security,income
138,2026-03-31,Lønoverførsel,"53,075.00",Income,"Pay, benefits and pension",income
160,2026-02-27,Lønoverførsel,"53,075.00",Income,"Pay, benefits and pension",income
177,2026-01-30,Lønoverførsel,"53,075.00",Income,"Pay, benefits and pension",income


In [13]:
show("refund")


=== refund  n=5  sum=6,680.00 ===


,date,text,amount,main_category,category,bucket
2,2026-08-12,MobilePay Boozt.com,"1,553.00",Income,Other income,refund
3,2026-08-12,MobilePay Boozt.com,"1,731.00",Income,Other income,refund
52,2026-07-09,MobilePay Boozt.com,899.00,Income,Other income,refund
68,2026-06-16,MobilePay Boozt.com,835.00,Income,Other income,refund
71,2026-06-03,MobilePay Boozt.com,"1,662.00",Income,Other income,refund


In [14]:
show("exclude_revolut_topup").head(20)


=== exclude_revolut_topup  n=70  sum=-231,014.00 ===


,date,text,amount,main_category,category,bucket
0,2026-08-24,VD REVOLUT**8260*,"-4,021.00",Loan and debt,Interest and fees,exclude_revolut_topup
1,2026-08-17,VD REVOLUT**8260*,"-4,392.00",Loan and debt,Interest and fees,exclude_revolut_topup
4,2026-08-11,VD REVOLUT**8260*,"-3,869.00",Loan and debt,Interest and fees,exclude_revolut_topup
7,2026-08-10,VD REVOLUT**8260*,"-5,000.00",Loan and debt,Interest and fees,exclude_revolut_topup
10,2026-08-06,VD REVOLUT**8260*,"-1,839.00",Loan and debt,Interest and fees,exclude_revolut_topup
13,2026-08-04,VD REVOLUT**8260*,-983.00,Loan and debt,Interest and fees,exclude_revolut_topup
14,2026-08-03,VD REVOLUT**8260*,"-3,023.00",Loan and debt,Interest and fees,exclude_revolut_topup
15,2026-08-03,VD REVOLUT**8260*,"-1,887.00",Loan and debt,Interest and fees,exclude_revolut_topup
18,2026-07-31,VD REVOLUT**8260*,"-2,831.00",Loan and debt,Interest and fees,exclude_revolut_topup
20,2026-07-31,VD REVOLUT**8260*,"-2,092.00",Loan and debt,Interest and fees,exclude_revolut_topup


In [15]:
# Savings moves both ways: negative = to savings, positive = from savings back to checking.
ops = df[df["bucket"].eq("exclude_savings")]
print("to savings:", f"{ops.loc[ops.amount < 0, 'amount'].sum():,.2f}")
print("from savings:", f"{ops.loc[ops.amount > 0, 'amount'].sum():,.2f}")
show("exclude_savings").head(15)

to savings: -392,032.00
from savings: 201,578.00

=== exclude_savings  n=58  sum=-190,454.00 ===


,date,text,amount,main_category,category,bucket
5,2026-08-11,Opsparingskonto,"4,923.00",Income,Other income,exclude_savings
8,2026-08-10,Opsparingskonto,"7,000.00",Income,Other income,exclude_savings
9,2026-08-10,Opsparingskonto,"1,574.00",Income,Other income,exclude_savings
12,2026-08-05,Opsparingskonto,"2,543.00",Income,Other income,exclude_savings
17,2026-08-03,Opsparingskonto,"-41,000.00",Savings and investment,Savings,exclude_savings
23,2026-07-29,Opsparingskonto,"3,024.00",Income,Other income,exclude_savings
26,2026-07-27,Opsparingskonto,"1,998.00",Income,Other income,exclude_savings
28,2026-07-23,Opsparingskonto,"1,432.00",Income,Other income,exclude_savings
32,2026-07-20,Opsparingskonto,"1,934.00",Income,Other income,exclude_savings
37,2026-07-17,Opsparingskonto,"4,315.00",Income,Other income,exclude_savings


In [16]:
show("exclude_internal")


=== exclude_internal  n=16  sum=89,643.61 ===


,date,text,amount,main_category,category,bucket
111,2026-04-30,Overførsel,"12,557.32",Income,Other income,exclude_internal
120,2026-04-21,Overførsel,"15,600.00",Income,Other income,exclude_internal
129,2026-04-13,5030 1360470,"-9,470.00",Other,Other expense,exclude_internal
133,2026-04-08,Overførsel,"16,500.00",Income,Other income,exclude_internal
142,2026-03-25,Overførsel,"28,400.00",Income,Other income,exclude_internal
144,2026-03-20,Overførsel,"24,000.00",Income,Other income,exclude_internal
147,2026-03-17,Overførsel,"19,500.00",Income,Other income,exclude_internal
158,2026-03-02,Mehdi Ordikhani,"-53,000.00",Other,Other expense,exclude_internal
170,2026-02-03,Overførsel,"11,900.00",Income,Other income,exclude_internal
180,2026-01-27,Overførsel,"2,600.00",Income,Other income,exclude_internal


## 9. Monthly view of proposed Jyske expenses

These are the amounts that would sit next to Revolut on the Annual tab if we keep the current `expense` bucket.

In [17]:
exp = df[df["bucket"].eq("expense")].copy()
exp["spend"] = exp["amount"].abs()
monthly = (
    exp.groupby(["month", "text"])["spend"]
    .sum()
    .unstack(fill_value=0.0)
    .sort_index()
)
monthly.loc["total"] = monthly.sum()
monthly

text,Akademikernes,BS JYSKE REALKREDIT,BS PARKERINGSLAUGET PARKKANTEN,BS PURE GYM DENMARK A/S,BS SKATTESTYRELSEN MOTOR OPKRÆVNING,Ida Div.Kontingent,Letsikring af barn ved dø,"Omkostninger, Netbank og Mobilbank",To Tryg Forsikring
month,,,,,,,,,
2026-01,0.00,0.00,240.08,389.00,0.00,0.00,856.08,100.00,0.00
2026-02,0.00,0.00,240.08,389.00,0.00,0.00,0.00,50.00,0.00
2026-03,0.00,"13,687.70",240.08,0.00,"1,630.00",0.00,0.00,50.00,0.00
2026-04,0.00,0.00,240.08,0.00,0.00,0.00,0.00,0.00,0.00
2026-05,0.00,0.00,240.08,0.00,0.00,878.25,0.00,2.00,0.00
2026-06,0.00,"13,674.71",147.14,0.00,0.00,0.00,0.00,0.00,0.00
2026-07,"1,551.00",0.00,224.59,0.00,0.00,0.00,0.00,56.00,"13,364.86"
2026-08,0.00,0.00,224.59,0.00,0.00,878.25,0.00,0.00,0.00
total,"1,551.00","27,362.41","1,796.72",778.00,"1,630.00","1,756.50",856.08,258.00,"13,364.86"


## Notes for the parser (later)

- File name in this export is `Mehdi Ordikhani_YYYY-MM-DD-YYYY-MM-DD.csv` — it does **not** contain `jyske`, so auto-detect will need a better rule.
- Amount format is English (`-4,021.00`), not Danish (`-4.021,00`).
- Do not use Jyske `MainCategory` as gospel: Revolut top-ups are tagged as interest/fees.
- Double-count risk: `VD REVOLUT**…` outflows must stay out of Jyske spend if Revolut card payments are already in the Revolut dashboard.
- Open questions in `review`: kids' Oskarkonto, Karoline home tax / MobilePay, passport/consulate, personal MobilePay.